# Model Training
Trains one LogisticRegression per pair on **in-sample data only** (≤ 2023-12-31).

The train/test split is a hard calendar date, not a row-fraction split.
Predictions are saved for the full period; the `split` column flags IS vs OOS.

In [1]:
import sys, numpy as np, pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
sys.path.append('..')

In [2]:
IS_END    = '2023-12-31'
OOS_START = '2024-01-01'

# Raw spread level excluded: non-stationary, different scale across pairs
ML_FEATURES = [
    'z_score', 'spread_change', 'spread_vol_20',
    'adf_p_252', 'hurst_252',
    'adf_stationary', 'hurst_mr', 'regime_score',
]

In [3]:
selected_pairs = pd.read_csv('../data_processed/selected_pairs.csv', index_col=0)
summary = []

for _, row in selected_pairs.iterrows():
    t1, t2  = row['ticker1'], row['ticker2']
    dataset = pd.read_csv(f'../data_processed/dataset_{t1}_{t2}.csv',
                          index_col=0, parse_dates=True)

    X, y  = dataset[ML_FEATURES], dataset['y']

    # IN-SAMPLE split (hard calendar boundary)
    X_is  = X.loc[:IS_END];   y_is  = y.loc[:IS_END]
    X_oos = X.loc[OOS_START:]; y_oos = y.loc[OOS_START:]

    model = Pipeline([
        ('scaler', StandardScaler()),
        ('lr',     LogisticRegression(max_iter=2000, C=0.1)),
    ])
    model.fit(X_is, y_is)   # trained on IS only

    # Predictions for full period (IS in-sample fit + OOS genuine forward prediction)
    pred = pd.DataFrame({
        'y':     y,
        'proba': model.predict_proba(X)[:, 1],
        'split': 'IS',
    }, index=X.index)
    pred.loc[OOS_START:, 'split'] = 'OOS'
    pred.to_csv(f'../data_processed/predictions_{t1}_{t2}.csv')

    oos_auc = (roc_auc_score(y_oos, model.predict_proba(X_oos)[:, 1])
               if len(y_oos.unique()) == 2 else np.nan)
    summary.append({
        'pair':     f'{t1}/{t2}',
        'IS rows':  len(X_is),
        'OOS rows': len(X_oos),
        'OOS AUC':  round(oos_auc, 4),
    })

results_df = pd.DataFrame(summary)
print(results_df.to_string(index=False))
print(f'\nMean OOS AUC: {results_df["OOS AUC"].mean():.4f}')
print('Model training complete.')

    pair  IS rows  OOS rows  OOS AUC
 EWJ/HYG      754       502   0.6010
 HYG/XLC      754       502   0.5387
 XLU/XOP      754       502   0.6515
HYG/MTUM      754       502   0.5702
 EWC/IWD      754       502   0.5443
GDXJ/LQD      754       502   0.5721
 USO/XLU      754       502   0.6134
 EWG/HYG      754       502   0.6811
 EWJ/XLC      754       502   0.6279
 EWC/XLF      754       502   0.6150
GDXJ/USO      754       502   0.5608
 XLP/XLV      754       502   0.6062
 XLP/XLU      754       502   0.6158
 EWA/XLB      754       502   0.5655
 GDX/USO      754       502   0.5578

Mean OOS AUC: 0.5948
Model training complete.
